# 09 - Monitoring Reports and Infrastructure Monitoring

This notebook extends the monitoring work completed in `08_monitoring.ipynb`.

Notebook 08 created CloudWatch prediction logs, custom monitoring metrics, PSI-based drift checks, a CloudWatch dashboard, and CloudWatch alarms.

This notebook creates formal report artifacts for:

- Model monitoring
- Data monitoring
- Infrastructure monitoring

The reports are saved locally and uploaded to S3 so they can be referenced in the project README, tracker, and final submission.

## Purpose

This notebook helps satisfy the following monitoring requirements:

1. Implement model monitors on the ML system
2. Implement data monitors on the ML system
3. Implement infrastructure monitors on the ML system
4. Create a monitoring dashboard for the ML endpoint/job on CloudWatch
5. Generate model and data monitoring reports

In [1]:
import json
from pathlib import Path
from datetime import datetime, timezone

import boto3
import pandas as pd

print("Imports ready")

Imports ready


In [2]:
with open("project_config.json") as f:
    cfg = json.load(f)

REGION = cfg["REGION"]
SOURCE_BUCKET = cfg["SOURCE_BUCKET"]

s3 = boto3.client("s3", region_name=REGION)

print("Region:", REGION)
print("Bucket:", SOURCE_BUCKET)

Region: us-east-1
Bucket: sagemaker-us-east-1-555419874521


In [3]:
MONITORING = cfg.get("MONITORING", {})

LOG_GROUP = MONITORING.get("LOG_GROUP", "/yelp-sentiment/predictions")
METRIC_NAMESPACE = MONITORING.get("METRIC_NAMESPACE", "YelpSentiment/Monitoring")
DASHBOARD_NAME = MONITORING.get("DASHBOARD_NAME", "yelp-sentiment-monitoring")
ALARMS = MONITORING.get("ALARMS", [
    "yelp-sentiment-low-confidence",
    "yelp-sentiment-feature-drift"
])

report_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%S")

print("Log group:", LOG_GROUP)
print("Metric namespace:", METRIC_NAMESPACE)
print("Dashboard:", DASHBOARD_NAME)
print("Alarms:", ALARMS)
print("Report timestamp:", report_timestamp)

Log group: /yelp-sentiment/predictions
Metric namespace: YelpSentiment/Monitoring
Dashboard: yelp-sentiment-monitoring
Alarms: ['yelp-sentiment-low-confidence', 'yelp-sentiment-feature-drift']
Report timestamp: 2026-05-31-21-08-34


## Create Monitoring Report Folder

This section creates a local folder for formal monitoring report artifacts. The reports will later be uploaded to S3 so they can be referenced in the README, tracker, or final project submission.

In [4]:
report_dir = Path("monitoring_reports")
report_dir.mkdir(exist_ok=True)

print("Report folder:", report_dir.resolve())

Report folder: /home/ec2-user/SageMaker/AAI540_Final-Project/monitoring_reports


## Model Monitoring Report

This report documents the model monitoring strategy implemented in Notebook 08.

The monitoring framework tracks:

- Prediction volume
- Positive prediction rate
- Mean prediction confidence
- Prediction logs stored in CloudWatch Logs

These signals help identify model degradation, abnormal prediction behavior, and operational issues.

In [5]:
model_monitoring_report = {
    "report_name": "Model Monitoring Report",
    "created_at_utc": report_timestamp,
    "project": "Yelp Review Sentiment Classification",

    "model_monitoring": {
        "prediction_volume": {
            "metric_name": "PredictionVolume",
            "description": "Number of predictions processed."
        },

        "positive_prediction_rate": {
            "metric_name": "PositivePredictionRate",
            "description": "Percentage of positive predictions."
        },

        "mean_confidence": {
            "metric_name": "MeanConfidence",
            "description": "Average prediction confidence."
        },

        "prediction_logs": {
            "cloudwatch_log_group": LOG_GROUP
        }
    },

    "cloudwatch_namespace": METRIC_NAMESPACE,

    "dashboard": DASHBOARD_NAME,

    "summary": (
        "Model monitoring is implemented through CloudWatch metrics and "
        "CloudWatch Logs. These metrics provide visibility into prediction "
        "behavior, confidence levels, and operational health."
    )
}

model_report_file = report_dir / "model_monitoring_report.json"

with open(model_report_file, "w") as f:
    json.dump(model_monitoring_report, f, indent=2)

print("Created:", model_report_file)

Created: monitoring_reports/model_monitoring_report.json


## Data Monitoring Report

This report documents the data monitoring strategy implemented in Notebook 08.

Data quality and feature drift are monitored using Population Stability Index (PSI).

The training dataset serves as the baseline distribution, while the production dataset serves as the current distribution.

PSI thresholds:

- PSI < 0.10 → No significant shift
- 0.10 ≤ PSI < 0.25 → Moderate shift
- PSI ≥ 0.25 → Significant shift

In [6]:
data_monitoring_report = {
    "report_name": "Data Monitoring Report",
    "created_at_utc": report_timestamp,
    "project": "Yelp Review Sentiment Classification",

    "baseline_dataset": "Training Split",
    "comparison_dataset": "Production Split",

    "monitoring_method": "Population Stability Index (PSI)",

    "features_monitored": cfg["FEATURE_COLS"],

    "drift_thresholds": {
        "psi_less_than_0_10": "No significant shift",
        "psi_0_10_to_0_25": "Moderate shift",
        "psi_greater_than_0_25": "Significant shift"
    },

    "cloudwatch_metric": "MaxFeaturePSI",

    "summary": (
        "Data monitoring is implemented using Population Stability Index "
        "(PSI) calculations. Feature distributions in production are "
        "compared against the training baseline. The maximum PSI value "
        "is published to CloudWatch and monitored through alarms."
    )
}

data_report_file = report_dir / "data_monitoring_report.json"

with open(data_report_file, "w") as f:
    json.dump(data_monitoring_report, f, indent=2)

print("Created:", data_report_file)

Created: monitoring_reports/data_monitoring_report.json


## Infrastructure Monitoring Report

This report documents infrastructure monitoring for the SageMaker and CloudWatch components used in the ML system.

Because the project uses SageMaker Batch Transform instead of a persistent real-time endpoint, there is no always-running endpoint to monitor.

Infrastructure monitoring focuses on:

- SageMaker training job status
- SageMaker batch transform job status
- CloudWatch metric publication
- CloudWatch dashboard availability
- CloudWatch alarm configuration

In [7]:
infrastructure_monitoring_report = {
    "report_name": "Infrastructure Monitoring Report",
    "created_at_utc": report_timestamp,
    "project": "Yelp Review Sentiment Classification",

    "infrastructure_components": {
        "sagemaker_training": {
            "component": "SageMaker SKLearn Training Job",
            "monitoring_focus": [
                "Training job completion status",
                "Training job failure detection",
                "Model artifact output location"
            ]
        },

        "sagemaker_batch_transform": {
            "component": "SageMaker Batch Transform",
            "monitoring_focus": [
                "Batch transform job completion status",
                "Batch transform output location",
                "Batch inference execution health"
            ]
        },

        "cloudwatch_logs": {
            "component": "CloudWatch Logs",
            "log_group": LOG_GROUP,
            "monitoring_focus": [
                "Prediction event logging",
                "Prediction-level traceability",
                "Recent prediction inspection"
            ]
        },

        "cloudwatch_metrics": {
            "component": "CloudWatch Custom Metrics",
            "namespace": METRIC_NAMESPACE,
            "metrics": [
                "PredictionVolume",
                "PositivePredictionRate",
                "MeanConfidence",
                "MaxFeaturePSI"
            ]
        },

        "cloudwatch_dashboard": {
            "dashboard_name": DASHBOARD_NAME,
            "description": "Dashboard used to monitor prediction behavior, model confidence, and feature drift."
        },

        "cloudwatch_alarms": {
            "alarms": ALARMS,
            "description": "Alarms monitor low model confidence and significant feature drift."
        }
    },

    "endpoint_monitoring_note": (
        "This project deploys the model using SageMaker Batch Transform rather than a persistent "
        "real-time endpoint. Therefore, infrastructure monitoring focuses on batch job status, "
        "CloudWatch logs, CloudWatch metrics, dashboards, and alarms instead of endpoint latency "
        "or invocation errors."
    ),

    "summary": (
        "Infrastructure monitoring is implemented through SageMaker job tracking and CloudWatch "
        "observability resources. This provides visibility into the health of training, batch "
        "inference, prediction logging, custom metrics, dashboards, and alarms."
    )
}

infra_report_file = report_dir / "infrastructure_monitoring_report.json"

with open(infra_report_file, "w") as f:
    json.dump(infrastructure_monitoring_report, f, indent=2)

print("Created:", infra_report_file)

Created: monitoring_reports/infrastructure_monitoring_report.json


## Upload Monitoring Reports to S3

Store monitoring reports in S3 so they can be referenced by the project team, README documentation, and final project submission.

In [8]:
report_prefix = f"monitoring-reports/{report_timestamp}"

report_files = [
    model_report_file,
    data_report_file,
    infra_report_file,
]

for report_file in report_files:
    s3_key = f"{report_prefix}/{report_file.name}"

    s3.upload_file(
        str(report_file),
        SOURCE_BUCKET,
        s3_key,
    )

    print(
        f"Uploaded: s3://{SOURCE_BUCKET}/{s3_key}"
    )

Uploaded: s3://sagemaker-us-east-1-555419874521/monitoring-reports/2026-05-31-21-08-34/model_monitoring_report.json
Uploaded: s3://sagemaker-us-east-1-555419874521/monitoring-reports/2026-05-31-21-08-34/data_monitoring_report.json
Uploaded: s3://sagemaker-us-east-1-555419874521/monitoring-reports/2026-05-31-21-08-34/infrastructure_monitoring_report.json


## Monitoring Report Summary

This notebook extends the monitoring implementation completed in Notebook 08.

### Monitoring Capabilities

#### Model Monitoring
- Prediction volume monitoring
- Positive prediction rate monitoring
- Mean confidence monitoring
- CloudWatch prediction logging

#### Data Monitoring
- Population Stability Index (PSI) drift detection
- Feature distribution monitoring
- Production vs training comparison
- CloudWatch drift metrics

#### Infrastructure Monitoring
- SageMaker training job monitoring
- SageMaker batch transform monitoring
- CloudWatch metrics
- CloudWatch alarms
- CloudWatch dashboard

### Generated Monitoring Artifacts

| Artifact | Purpose |
|-----------|-----------|
| model_monitoring_report.json | Documents model monitoring strategy |
| data_monitoring_report.json | Documents data drift monitoring strategy |
| infrastructure_monitoring_report.json | Documents infrastructure monitoring strategy |

### Cloud Resources

- CloudWatch Dashboard
- CloudWatch Logs
- CloudWatch Custom Metrics
- CloudWatch Alarms
- Monitoring Reports stored in S3

This notebook completes the monitoring and reporting layer of the Yelp Sentiment MLOps workflow.

In [9]:
print("=" * 70)
print("YELP SENTIMENT MLOPS MONITORING SUMMARY")
print("=" * 70)

print("\nMonitoring Reports Created:")
print("  • model_monitoring_report.json")
print("  • data_monitoring_report.json")
print("  • infrastructure_monitoring_report.json")

print("\nS3 Report Location:")
print(f"  s3://{SOURCE_BUCKET}/{report_prefix}/")

print("\nMonitoring Components:")
print("  • CloudWatch Logs")
print("  • CloudWatch Metrics")
print("  • CloudWatch Dashboard")
print("  • CloudWatch Alarms")
print("  • PSI Drift Monitoring")

print("\nStatus: COMPLETE")
print("=" * 70)

YELP SENTIMENT MLOPS MONITORING SUMMARY

Monitoring Reports Created:
  • model_monitoring_report.json
  • data_monitoring_report.json
  • infrastructure_monitoring_report.json

S3 Report Location:
  s3://sagemaker-us-east-1-555419874521/monitoring-reports/2026-05-31-21-08-34/

Monitoring Components:
  • CloudWatch Logs
  • CloudWatch Metrics
  • CloudWatch Dashboard
  • CloudWatch Alarms
  • PSI Drift Monitoring

Status: COMPLETE
